In [2]:
import random

def get_dihedral_commutes(a_type, a_val, b_type, b_val, n):
    """二面体群の2つの要素が可換かどうかを判定する"""
    nh = n // 2
    if a_type == 0 and b_type == 0: return True  # 回転同士は常に可換
    if a_type == 0 and b_type == 1: return (a_val % nh == 0) # 回転が中心に属せば可換
    if a_type == 1 and b_type == 0: return (b_val % nh == 0)
    if a_type == 1 and b_type == 1: return ((a_val - b_val) % nh == 0) # 反転同士の可換条件
    return False

def check_girth_6(elements, matrix_shape, n):
    """2x3のブロック行列において長さ4のサイクル（girth 4）がないか確認する"""
    r, c = matrix_shape
    for r1 in range(r):
        for r2 in range(r1 + 1, r):
            for c1 in range(c):
                for c2 in range(c1 + 1, c):
                    # パス: E1 -> E2^-1 -> E3 -> E4^-1
                    e1, e2, e3, e4 = elements[r1*c+c1], elements[r1*c+c2], elements[r2*c+c2], elements[r2*c+c1]
                    
                    # 群演算のシミュレーション
                    curr_t, curr_v = e1
                    # * inv(e2)
                    t2, v2 = e2
                    curr_v = (curr_v - v2) % n if t2 == 0 else ((-curr_v + v2) % n)
                    curr_t = (curr_t + t2) % 2
                    # * e3
                    t3, v3 = e3
                    curr_v = (curr_v + v3) % n if t3 == 0 else ((-curr_v + v3) % n)
                    curr_t = (curr_t + t3) % 2
                    # * inv(e4)
                    t4, v4 = e4
                    curr_v = (curr_v - v4) % n if t4 == 0 else ((-curr_v + v4) % n)
                    curr_t = (curr_t + t4) % 2
                    
                    if curr_t == 0 and curr_v == 0: return 4
    return 6

def find_apm_parameters(L_half=6, P=768):
    n = P // 2
    nh = n // 2
    f_types = [1, 1, 0, 0, 0, 0] # F0, F1は反転(s)を含む
    g_types = [0, 0, 1, 1, 0, 0] # G2, G3は反転(s)を含む
    
    while True:
        f_vals = [random.randint(0, n-1) for _ in range(L_half)]
        g_vals = [random.randint(0, n-1) for _ in range(L_half)]
        
        # 条件Aのための強制制約
        for j in [0, 1, 4, 5]: g_vals[j] = random.choice([0, nh])
        for i in [2, 3, 4, 5]: f_vals[i] = random.choice([0, nh])
        f_vals[0] = (g_vals[2] + random.choice([0, nh])) % n # (0,2)を可換に
        f_vals[1] = (g_vals[3] + random.choice([0, nh])) % n # (1,3)を可換に
        
        # 条件B: 非可換性のチェック
        if get_dihedral_commutes(f_types[0], f_vals[0], g_types[3], g_vals[3], n): continue
        if get_dihedral_commutes(f_types[1], f_vals[1], g_types[2], g_vals[2], n): continue
        
        # 条件C: サイクルチェック (2x3 ブロック行列を想定)
        if check_girth_6(list(zip(f_types, f_vals)), (2, 3), n) < 6: continue
        if check_girth_6(list(zip(g_types, g_vals)), (2, 3), n) < 6: continue
        
        return f_types, f_vals, g_types, g_vals

# 実行と結果表示
f_t, f_v, g_t, g_v = find_apm_parameters()
print(f"--- F_i (i=0..5) ---")
for i in range(6):
    prefix = "s * r^" if f_t[i] else "r^"
    print(f"F{i}: {prefix}{f_v[i]}")

print(f"\n--- G_j (j=0..5) ---")
for j in range(6):
    prefix = "s * r^" if g_t[j] else "r^"
    print(f"G{j}: {prefix}{g_v[j]}")

--- F_i (i=0..5) ---
F0: s * r^345
F1: s * r^381
F2: r^0
F3: r^192
F4: r^0
F5: r^0

--- G_j (j=0..5) ---
G0: r^192
G1: r^0
G2: s * r^345
G3: s * r^189
G4: r^192
G5: r^0
